<a href="https://colab.research.google.com/github/ges45-learn/ch2_contd/blob/wk1-assignment1/Copy_of_SCS_Colab_Notebook_W6_%5B056_Document_SCS%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course notebook notes

Throughout this notebook you will be taught how to work with Colab and OpenLane. More specifically, you will use this Notebook to:

* compare IC designs from 1970s and 2020s  
* explore a basic semiconductor layout, identifying the materials that are included in it
* study and characterise a given standard library cell.
* look at a design tool, explore it and consider what it is doing
* look at a design tool, inserting input/output (I/O) pads for wiring bonding and experience verification

**NOTE 1:** In module 1, we asked you to create a Google and Colab account. For optimal experience, you must be logged into the Google account that you created every time you work with this file.

**NOTE 2:** If you have not done so already, be sure to make a copy of this notebook in your drive. This will help you retain any changes you make and ensure a smooth journey through the following sections.

---

# Imports, authentication and preparation
Running the following cells will import all the data we need for this notebook to work. Simply select the code cell below and it will import all of these important packages.

Note that you will need to run these every time you open this notebook. Be sure to complete this step before attempting any of the tasks in this notebook.

Setting up the OpenLane runtime environment

For this to work, we want to:
* install OpenLane (v2) and its dependencies
* download and set up [open source sky130 PDK](https://github.com/google/skywater-pdk/) by Google and Skywater.

Created by Efabless, *edited by Dr Matthew Tang*.


In [ ]:
# @title Setup Nix {display-mode: "form"}
# @markdown Nix is a package manager with an emphasis on reproducible builds,
# @markdown     and it is the primary method for installing OpenLane2.
# @markdown
# @markdown     This step installs the Nix package manager and enables the
# @markdown     experimental 'flakes' feature.
# @markdown
# If you're not in a Colab, this just sets the environment variables.
# You will need to install Nix and enable flakes on your own following
# [this guide](https://openlane2.readthedocs.io/en/stable/getting_started/common/nix_installation/index.html).
import os
import sys
import shutil

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        !curl -L https://nixos.org/nix/install | bash -s -- --daemon --yes
        !echo "extra-experimental-features = nix-command flakes" >> /etc/nix/nix.conf
        !killall nix-daemon
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

In [ ]:
# @title Get OpenLane2 and SKY130 Open PDK {"display-mode":"form"}
# @markdown Click the ▷ button to download and install OpenLane.
# @markdown
# @markdown     This will install OpenLane's tool dependencies using Nix,
# @markdown     and OpenLane itself using PIP.
# @markdown
# @markdown     Note that `python3-tk` may need to be installed using your OS's
# @markdown     package manager.
import os
import subprocess
import IPython

openlane_version = "main"  # @param {key:"OpenLane Version", type:"string"}

if openlane_version == "latest":
    openlane_version = "main"

pdk_root = "~/.volare"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

openlane_ipynb_path = os.path.join(os.getcwd(), "openlane_ipynb")

display(IPython.display.HTML("<h3>Downloading OpenLane…</a>"))

TESTING_LOCALLY = False
!rm -rf {openlane_ipynb_path}
!mkdir -p {openlane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {openlane_ipynb_path}
else:
    !curl -L "https://github.com/efabless/openlane2/tarball/{openlane_version}" | tar -xzC {openlane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading OpenLane's dependencies…</a>"))
try:
    subprocess.check_call(
        ["nix", "profile", "install", ".#colab-env", "--accept-flake-config"],
        cwd=openlane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=openlane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))

#pdk_tmp = user_workspace_path + '/.volare'
#if os.path.exists(pdk_tmp):
#  IPython.display.HTML('<p>PDK found in user workspace. Linking ...</p>')


import volare

volare.enable(
    volare.get_volare_home(pdk_root),
    pdk,
    open(
        os.path.join(openlane_ipynb_path, "openlane", "open_pdks_rev"),
        encoding="utf8",
    )
    .read()
    .strip(),
)
# create symlink for PDK
#!ln -s {pdk_root} ~/.volare
sys.path.insert(0, openlane_ipynb_path)

display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

# link the latest versions with the assumed directory name
!cd /root/.volare/volare/sky130/versions && ln -s `cat ../current` bdc9412b3e468c102d01b7cf6337be06ec6e9c9a

## Quick check

After running the above cells, you should see the version number of OpenLane. If you see that, you are ready to run the tasks below.

In [ ]:
import openlane
print(openlane.__version__)

## Utility functions

These functions are required to generate high-quality images from the OpenLane's outputs (GDSII layout). Execute the code box once before beginning any tasks in the course.

[1] Create images from GDSII layout (`gds_render.py`).

This short Python program generates a high-resolution image from the GDSII physical layout provided, then inspect the images closely in your browser. (You may download the image too).

Recommended resolution:

Module 1: 2,000 pixels

Other modules: 1,000 pixels

In [ ]:
%%writefile /content/gds_render.py
# @markdown You may adjust the resolution of the image by modifying the value below. The default is 1,000 pixels.
#!/usr/bin/env python3
# Copyright (c) 2021-2022 Efabless Corporation
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Original Copyright Follows
#
# BSD 3-Clause License
#
# Copyright (c) 2018, The Regents of the University of California
# All rights reserved.
#
# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions are met:
#
# * Redistributions of source code must retain the above copyright notice, this
#   list of conditions and the following disclaimer.
#
# * Redistributions in binary form must reproduce the above copyright notice,
#   this list of conditions and the following disclaimer in the documentation
#   and/or other materials provided with the distribution.
#
# * Neither the name of the copyright holder nor the names of its
#   contributors may be used to endorse or promote products derived from
#   this software without specific prior written permission.
#
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

import pya
import sys

def render(
    input: str,
    output: str,
):
    PDK = '~/.volare/volare/sky130/versions/bdc9412b3e468c102d01b7cf6337be06ec6e9c9a'
    lyt = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'
    lyp = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyp'
    lym = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'

    img_res = 1000 # @param {key:"image resolution", type:"number"}
    print(f"image resolution: {img_res}")
    # display ONLY the layers in the following list
    selected_layers = [
        "diff.",
        "dnwell.",
        "li1.",
        "licon1.",
        "mcon.",
        "met1.",
        "met2.",
        "met3.",
        "met4.",
        "met5.",
        #"vnpc.",
        #"vnsdm.",
        #"vnwell.",
        "poly.",
        "psdm.",
        "pwell.",
        "tap.",
        "via.",
        "via2.",
        "via3.",
        "via4."
    ]

    try:
        gds = input.endswith(".gds")

        # Load technology file
        tech = pya.Technology()
        tech.load(lyt)

        layout_options = None

        view = pya.LayoutView()
        view.load_layer_props(lyp)

        if gds:
            view.load_layout(input)
        else:
            view.load_layout(input, layout_options, lyt)

        view.max_hier()

        # control the visibility of the layers
        for layer in view.each_layer():
            layer.visible = False
            if '122/16@1' in layer.source: # grid and ruler
                layer.visible = True
            for t in selected_layers:
                if layer.name.startswith(t):
                    layer.visible = True

        pixels = view.get_pixels_with_options(img_res, img_res)

        with open(output, "wb") as f:
            f.write(pixels.to_png_data())

    except Exception as e:
        print(e)
        exit(1)


if __name__ == "__main__":
    gds_file = ''
    if len(sys.argv) >= 2:
        gds_file = sys.argv[1]
    else:
        quit('Please provide a GDS file.')

    out_file = 'output.png'
    if len(sys.argv) >= 3:
        out_file = sys.argv[2]
    print(f"Input GDS file: {gds_file}")
    print(f"Output PNG file: {out_file}")

    render(gds_file, out_file)
    print("done.")

[2] Install a customised render function.

In [ ]:
%%writefile /content/openlane_ipynb/openlane/scripts/klayout/render.py
#!/usr/bin/env python3
# @markdown This makes ``display()`` to generate an additional png image (``/content/openlane_run/output.png``) of a higher resolution (default: 5000 pixels).
# More information about layers can be found at this
# [link](https://skywater-pdk.readthedocs.io/en/main/rules/layers.html). To hide the layer in the output image, comment off from the list.
#
# Copyright (c) 2021-2022 Efabless Corporation
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from typing import Tuple
import pya
import click
@click.command()
@click.option("-o", "--output", required=True)
@click.option("-l", "--input-lef", "input_lefs", multiple=True,)
@click.option( "-T", "--lyt", required=True, help="KLayout .lyt file",)
@click.option( "-P", "--lyp", required=True, help="KLayout .lyp file",)
@click.option( "-M", "--lym", required=True, help="KLayout .map (LEF/DEF layer map) file",)
@click.argument("input")
def render(input_lefs: Tuple[str, ...], output: str, lyt: str, lyp: str, lym: str, input: str,):
    image_res = 5000 # output image resolution = 5000 pixels
    # display ONLY the layers in the following list
    selected_layers = [
        "diff.",
        "dnwell.",
        "li1.",
        "licon1.",
        "mcon.",
        "met1.",
        "met2.",
        "met3.",
        "met4.",
        "met5.",
        #"vnpc.",
        #"vnsdm.",
        #"vnwell.",
        "poly.",
        "psdm.",
        "pwell.",
        "tap.",
        "via.",
        "via2.",
        "via3.",
        "via4."
    ]

    try:
        gds = input.endswith(".gds")

        # Load technology file
        tech = pya.Technology()
        tech.load(lyt)

        layout_options = None
        if not gds:
            layout_options = tech.load_layout_options
            layout_options.lefdef_config.map_file = lym
            layout_options.lefdef_config.macro_resolution_mode = 1
            layout_options.lefdef_config.read_lef_with_def = False
            layout_options.lefdef_config.lef_files = list(input_lefs)

        view = pya.LayoutView()
        view.load_layer_props(lyp)

        if gds:
            view.load_layout(input)
        else:
            view.load_layout(input, layout_options, lyt)

        view.max_hier()

        # control the visibility of the layers
        for layer in view.each_layer():
            layer.visible = False
            if '122/16@1' in layer.source: # grid and ruler
                layer.visible = True
            for t in selected_layers:
                if t in layer.name:
                    layer.visible = True

        pixels = view.get_pixels_with_options(1000, 1000)

        with open(output, "wb") as f:
            f.write(pixels.to_png_data())

        # save an image of larger resolution
        pixels = view.get_pixels_with_options(image_res, image_res)
        with open("/content/openlane_run/output.png", "wb") as f:
            f.write(pixels.to_png_data())

    except Exception as e:
        print(e)
        exit(1)

if __name__ == "__main__":
    render()

Now you may proceed with the actual task for the module.


---



# Module 1: Introduction to semiconductor industry

This is the first task you will complete in this Colab notebook. Before proceeding, please ensure that you are working in your own copy of the notebook and you have successfully run all the cells above.

In this first task, we would like you to compare two chip designs and apply Moore's Law to work out their reasonable sizes.

---
**What you need to do**

Firstly, set up Nix, OpenLane2 and Sky130 Open PDK in your Colab runtime environment, and then action the following steps:
1. Download a zip file of prepared sample files.
1. Generate an image of chip layout of design 1 - CD4011.
1. Generate an image of chip layout of design 2 - Picorv32.
1. Apply Moore's law to scale the sizes of the chips to the respective era.

The following sections contain the code and packages you need to complete these steps.

## Step 1: Download sample files

To get started, download a copy of the prepared sample files from the given Google drive link. The downloaded file will be named ``M1_set.tar.gz`` and extracted into your working directory ``/content``.

Run the following code box.

In [ ]:
!pip install gdown
import gdown
url = 'https://drive.google.com/file/d/1CtEPSrDfIwBzrcLgSKs793dmcBjf1aLV/view?usp=share_link'
output_path = 'M1_set.tar.gz'
gdown.download(url, output_path, quiet=False, fuzzy=True)
!tar zxvf M1_set.tar.gz

In [ ]:
%%writefile /content/gds_render.py
# @markdown Adjust the resolution of the generated image.
# Copyright (c) 2021-2022 Efabless Corporation
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Original Copyright Follows
#
# BSD 3-Clause License
#
# Copyright (c) 2018, The Regents of the University of California
# All rights reserved.
#
# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions are met:
#
# * Redistributions of source code must retain the above copyright notice, this
#   list of conditions and the following disclaimer.
#
# * Redistributions in binary form must reproduce the above copyright notice,
#   this list of conditions and the following disclaimer in the documentation
#   and/or other materials provided with the distribution.
#
# * Neither the name of the copyright holder nor the names of its
#   contributors may be used to endorse or promote products derived from
#   this software without specific prior written permission.
#
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

import pya
import sys

def render(
    input: str,
    output: str,
):
    PDK = '~/.volare/volare/sky130/versions/bdc9412b3e468c102d01b7cf6337be06ec6e9c9a'
    lyt = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'
    lyp = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyp'
    lym = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'

    img_res = 2000 # @param {key:"image resolution", type:"number"}
    print(f"image resolution: {img_res}")
    # display ONLY the layers in the following list
    selected_layers = [
        "diff.",
        "dnwell.",
        "li1.",
        "licon1.",
        "mcon.",
        "met1.",
        "met2.",
        "met3.",
        "met4.",
        "met5.",
        #"vnpc.",
        #"vnsdm.",
        #"vnwell.",
        "poly.",
        "psdm.",
        "pwell.",
        "tap.",
        "via.",
        "via2.",
        "via3.",
        "via4."
    ]

    try:
        gds = input.endswith(".gds")

        # Load technology file
        tech = pya.Technology()
        tech.load(lyt)

        layout_options = None

        view = pya.LayoutView()
        view.load_layer_props(lyp)

        if gds:
            view.load_layout(input)
        else:
            view.load_layout(input, layout_options, lyt)

        view.max_hier()

        # control the visibility of the layers
        for layer in view.each_layer():
            layer.visible = False
            if '122/16@1' in layer.source: # grid and ruler
                layer.visible = True
            for t in selected_layers:
                if layer.name.startswith(t):
                    layer.visible = True

        pixels = view.get_pixels_with_options(img_res, img_res)

        with open(output, "wb") as f:
            f.write(pixels.to_png_data())

    except Exception as e:
        print(e)
        exit(1)


if __name__ == "__main__":
    gds_file = ''
    if len(sys.argv) >= 2:
        gds_file = sys.argv[1]
    else:
        quit('Please provide a GDS file.')

    out_file = 'output.png'
    if len(sys.argv) >= 3:
        out_file = sys.argv[2]
    print(f"Input GDS file: {gds_file}")
    print(f"Output PNG file: {out_file}")

    render(gds_file, out_file)
    print("done.")

## Step 2: Explore Design #1 - CD4011 Quad-NAND gate IC (1970)

Next, we will start visualising these files through Colab. We'll start with a design from 1970, the CD4011 Quad-NAND gate IC. Run the code cells below to generate a visual of it.

The GDSII (Graphic Design System II) stream format is a database file format used as the de facto standard for data exchange in the semiconductor industry. It is a binary file that stores the full layout of an integrated circuit (IC) as a collection of planar geometric shapes. In this case, ``cds4011_quadnand.gds`` is the GDSII layout.



In [ ]:
!python gds_render.py cd4011/runs/RUN_sample/final/gds/cds4011_quadnand.gds cd4011_gds.png

As you have probably noticed, there is now an image file with the above details, sitting within the Files tab of your notebook. You will find it when you select the Folder Icon on the top left sidebar.

![image](https://drive.google.com/uc?export=view&id=1xKoMIelkH968WHdPywZRVvmWrx0EIWHp)

Navigate to the image file and double click it to open within Colab. If you would like to, you also have the option to download it by selecting the three dots next to it and then choosing 'Download'.

![image](https://drive.google.com/uc?export=view&id=1X79ZxOkXnPQQc2J8HawPHA37q6rEw27O)

Now go ahead and inspect the image generated from the physical layout of CD4011 Quad-NAND gate design. Aim to answer the following:

1. How many input/output pins are there? They would appear to be the **legs** around the layout.

*Enter your answer here.*

2. Estimate the die area (to the nearest micron$^2$).

*Enter your answer here.*

## Step 3: Explore Design #2 - Picorv32 32-bit RISC-V microprocessor (2020)

Next, we will continue visualising files. We will look at a more recent example from 2020. To see the Picorv32 32-bit RISC-V microprocessor, run the code cells below.

In [ ]:
!python gds_render.py picorv32a/runs/RUN_sample/final/gds/picorv32.gds picorv32_gds.png

As with the previous visualisation, you will find a file with the details listed above. Navigate to the Files tab to open or download it.

Now go ahead and inspect the image generated from the physical layout of Pico32rv 32-bit microprocessor design design. Aim to answer the following:

1. How many input/output pins are there?

*Enter your answer here.*

2.  Estimate the die area (to the nearest micron$^2$).

*Enter your answer here.*

## Step 4: Applying Moore's Law

The two sample chips were implemented using the same technology from SKY130 PDK (180nm), which would be the popular technology node in the year 2000.

Assume that CD4011 and Pico32rv were designed in 1970 and 2020 respectively, calculate the probable physical sizes of the dies based on Moore's Law which says that the area of a die reduces by 50% every two years.

1. Probable size of CD4011 in 1970

*Enter your answer here.*

2. Probable size of Picorv32 in 2020

*Enter your answer here.*

3. The ratio between the sizes of Picorv32 to CDS4011

*Enter your answer here.*

In [ ]:
#use this: https://numpy.org/numpy-tutorials/mooreslaw-tutorial/

If you compare the complexity of the circuits and the actual physical areas of the die, you will see the rapid advances of semiconductor technologies in the past 70 years.


---
**What did I learn?**

When you have completed exploring the visuals above, add your thoughts in response to the question below. You will want to keep these, as you will find them useful when working on later modules and your final assignment. To get started, double click this cell and edit the text.

**What would be the challenges for design, manufacturing and assembly for the Design #2 Picorv32 microprocessor?**

*Enter your answer here.*


---
You have now reached the end of the notebook for this week. Please return to Canvas to complete the rest of the work in module 1.

We will resume our work in this notebook next week, when we will be working through Module 2: Semiconductor materials and their properties.

#Module 2: Semiconductor materials and their properties

In this section you will work on your second module task. As you did before, please ensure that you are working in your own copy of the notebook and you have successfully run all the cells above.

In this task, we we will explore the colouring scheme used by the [klayout layout viewer](https://www.klayout.de/). Each material in the IC fabrication is presented by a specific colour and pattern to help the IC designer to understand and check the layout more easily.

---

**What you need to do**

Firstly, set up Nix, OpenLane2 and SKY130. Open PDK in your Colab runtime environment, and then action the following steps:
1. Download the prepared sample files.
1. Inspect the semiconductor materials in the layout of the NAND gate.
1. Explore the relationships between different semiconductor materials in detail.

The following sections contain the code and packages you need to complete these steps.

## Step 1: Download sample files

To get started, download a copy of the colour key to the material and the GDSII layout of a 2-input NAND gate (used in CD4011 in module 1) from the given Google drive links. The downloaded files will be named ``klayout_layer_key.png`` and ``nand2_1.gds`` into your working directory ``/content``.

Run the following code boxes.

In [ ]:
import gdown
url = 'https://drive.google.com/file/d/1mZjzWJh4J6Xi1ObvVk4izvUGQk3DICKQ/view?usp=share_link'
output_path = 'klayout_layer_key.png'
gdown.download(url, output_path, quiet=False, fuzzy=True)
url = 'https://drive.google.com/file/d/1ghqBzj1cdFdujhDbXK1WJEYo1o2_1JO-/view?usp=share_link'
output_path = 'nand2_1.gds'
gdown.download(url, output_path, quiet=False, fuzzy=True)

## Step 2: Inspect the materials used in the GDSII layout

In [ ]:
%%writefile /content/gds_render.py
# @markdown Adjust the resolution of the image to 1000 pixels by 1000 pixels.
# Copyright (c) 2021-2022 Efabless Corporation
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Original Copyright Follows
#
# BSD 3-Clause License
#
# Copyright (c) 2018, The Regents of the University of California
# All rights reserved.
#
# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions are met:
#
# * Redistributions of source code must retain the above copyright notice, this
#   list of conditions and the following disclaimer.
#
# * Redistributions in binary form must reproduce the above copyright notice,
#   this list of conditions and the following disclaimer in the documentation
#   and/or other materials provided with the distribution.
#
# * Neither the name of the copyright holder nor the names of its
#   contributors may be used to endorse or promote products derived from
#   this software without specific prior written permission.
#
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

import pya
import sys

def render(
    input: str,
    output: str,
):
    PDK = '~/.volare/volare/sky130/versions/bdc9412b3e468c102d01b7cf6337be06ec6e9c9a'
    lyt = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'
    lyp = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyp'
    lym = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'

    img_res = 1000 # @param {key:"image resolution", type:"number"}
    print(f"image resolution: {img_res}")
    # display ONLY the layers in the following list
    selected_layers = [
        "diff.",
        "dnwell.",
        "li1.",
        "licon1.",
        "mcon.",
        "met1.",
        "met2.",
        "met3.",
        "met4.",
        "met5.",
        #"vnpc.",
        #"vnsdm.",
        #"vnwell.",
        "poly.",
        "psdm.",
        "pwell.",
        "tap.",
        "via.",
        "via2.",
        "via3.",
        "via4."
    ]

    try:
        gds = input.endswith(".gds")

        # Load technology file
        tech = pya.Technology()
        tech.load(lyt)

        layout_options = None

        view = pya.LayoutView()
        view.load_layer_props(lyp)

        if gds:
            view.load_layout(input)
        else:
            view.load_layout(input, layout_options, lyt)

        view.max_hier()

        # control the visibility of the layers
        for layer in view.each_layer():

            layer.visible = False
            if '122/16' in layer.source: # grid and ruler
                layer.visible = True
            for t in selected_layers:
                if layer.name.startswith(t):
                    layer.visible = True

        pixels = view.get_pixels_with_options(img_res, img_res)

        with open(output, "wb") as f:
            f.write(pixels.to_png_data())

    except Exception as e:
        print(e)
        exit(1)


if __name__ == "__main__":
    gds_file = ''
    if len(sys.argv) >= 2:
        gds_file = sys.argv[1]
    else:
        quit('Please provide a GDS file.')

    out_file = 'output.png'
    if len(sys.argv) >= 3:
        out_file = sys.argv[2]
    print(f"Input GDS file: {gds_file}")
    print(f"Output PNG file: {out_file}")

    render(gds_file, out_file)
    print("done.")

Now, run the utility function to generate an image of the layout of 2-input NAND gate.

In [ ]:
!python gds_render.py nand2_1.gds nand2_1.png

Download and view the generated file ``nand2_1.png``.

Count how many materials (each has a unique colour and pattern) are there in this picture.

List three kinds of materials that you recognise using the key ``klayout_layer_key.png``.

*Enter your answer here*

## Step 3: Explore semiconductor materials

Look more closely at the generated picture ``nand2_1.png``.

The black background represents the silicon substrate (wafer). Some areas of the wafer will be doped to n-type and p-type silicon, which are known as **diffusion** (diff).

1. What are the approximate dimension (in pixels) of two areas of diffusion?

*Enter your answer here*

The top diffusion is inside the purple dotted area of P+ source/drain implant (psdm), which means that the diffusion is strongly doped p-type material. The bottom diffusion is otherwise doped n-type material.

Above the diffusion, the next layer is the **polysilicon** (poly). Of course, there will be a layer of oxide as insulator between the poly and substrate but it is implicit in the layout.

2. How many overlapping areas are there between **poly** and **diff**?

*Enter your answer here*

Each of these areas is a transistor - either NMOS (when poly over n-type diffusion) or PMOS (when poly over p-type diffusion).

3. What is the ratio between the size of NMOS to PMOS? What would be a reason for using different sizes of transistors? (*Hint: consider the carrier mobility of semiconductors*)

*Enter your answer here*

Above polysilicon, there are two layers of interconnects - wiring materials like aluminium or copper. To make an electrical connection, a via is made between the layers and filled with metal like tungsten.

4. Focusing on the layer **local interconnect** (li), how many **licon1** connections are there between:

li and poly?: *Enter your answer here*

li and diff?: *Enter your answer here*

Lastly, there are two **metal** (metal1) stripes across the top and the bottom of the cell.

5. Could you suggest a usage of these two stripes?

*Enter your answer here*


By now, you should be more familiar with the colouring scheme in klayout and be able to explore the relationship between various materials in a chip layout.

---
**What did I learn?**

When you have completed the exploration of semiconductor materials in a chip layout, add your thoughts in response to the following question. You will want to keep these, as you will find them useful when working on later modules and your final assignment. To get started, double click this cell and edit the text.

**What would be the challenges to fabricate the exact geometries of the layout as the technology nodes scale down to as small as a few nanometres?**

*Enter your answer here.*

---
You have now reached the end of the notebook for this week. Please return to Canvas to complete the rest of the work in module 2.

We will resume our work in this notebook next week, when we will be working through Module 3: Creating circuits from materials.

# Module 3: Creating circuits from materials

In this section you will work on your third module task. As you did before, please ensure that you are working in your own copy of the notebook and you have successfully run all the cells above.

In this task, we will study a secret standard cell from the SKY130 PDK and deduce its logic function. You will find your knowledge about GDSII layout from the practical task in module 2 and CMOS transistors useful.

---

**What you need to do**

Firstly, set up Nix, OpenLane2 and SKY130 Open PDK in your Colab runtime environment, and then action the following steps:
1. Download the prepared sample files.
1. Extract basic information about the standard cell.
1. Identify logic function of the secret gate.
1. Consider the challenge of standard cell design.

The following sections contain the code and packages you need to complete these steps.

## Step 1: Download sample files

To get started, download a copy of the colour key to the material and the GDSII layout of a secret standard cell from the given Google drive links. The downloaded files will be named ``klayout_layer_key.png`` and ``secret.gds`` into your working directory ``/content``.

Run the following code boxes.

In [ ]:
import gdown
url = 'https://drive.google.com/file/d/1mZjzWJh4J6Xi1ObvVk4izvUGQk3DICKQ/view?usp=share_link'
output_path = 'klayout_layer_key.png'
gdown.download(url, output_path, quiet=False, fuzzy=True)
url = 'https://drive.google.com/file/d/1Dva_ebQGLb0ww6sjBNxJlq5my72bDsmw/view?usp=share_link'
output_path = 'secret.gds'
gdown.download(url, output_path, quiet=False, fuzzy=True)

## Step 2: Extract basic information about the standard cell

In [ ]:
%%writefile /content/gds_render.py
# @markdown Adjust the resolution of the image to 1000 pixels by 1000 pixels
# Copyright (c) 2021-2022 Efabless Corporation
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Original Copyright Follows
#
# BSD 3-Clause License
#
# Copyright (c) 2018, The Regents of the University of California
# All rights reserved.
#
# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions are met:
#
# * Redistributions of source code must retain the above copyright notice, this
#   list of conditions and the following disclaimer.
#
# * Redistributions in binary form must reproduce the above copyright notice,
#   this list of conditions and the following disclaimer in the documentation
#   and/or other materials provided with the distribution.
#
# * Neither the name of the copyright holder nor the names of its
#   contributors may be used to endorse or promote products derived from
#   this software without specific prior written permission.
#
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
# AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
# DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
# FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
# DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
# SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
# CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
# OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
# OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

import pya
import sys

def render(
    input: str,
    output: str,
):
    PDK = '~/.volare/volare/sky130/versions/bdc9412b3e468c102d01b7cf6337be06ec6e9c9a'
    lyt = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'
    lyp = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyp'
    lym = f'{PDK}/sky130A/libs.tech/klayout/tech/sky130A.lyt'

    img_res = 1000 # @param {key:"image resolution", type:"number"}
    print(f"image resolution: {img_res}")
    # display ONLY the layers in the following list
    selected_layers = [
        "diff.",
        "dnwell.",
        "li1.",
        "licon1.",
        "mcon.",
        "met1.",
        "met2.",
        "met3.",
        "met4.",
        "met5.",
        #"vnpc.",
        #"vnsdm.",
        #"vnwell.",
        "poly.",
        "psdm.",
        "pwell.",
        "tap.",
        "via.",
        "via2.",
        "via3.",
        "via4."
    ]

    try:
        gds = input.endswith(".gds")

        # Load technology file
        tech = pya.Technology()
        tech.load(lyt)

        layout_options = None

        view = pya.LayoutView()
        view.load_layer_props(lyp)

        if gds:
            view.load_layout(input)
        else:
            view.load_layout(input, layout_options, lyt)

        view.max_hier()

        # control the visibility of the layers
        for layer in view.each_layer():
            layer.visible = False
            if '122/16@1' in layer.source: # grid and ruler
                layer.visible = True
            for t in selected_layers:
                if layer.name.startswith(t):
                    layer.visible = True

        pixels = view.get_pixels_with_options(img_res, img_res)

        with open(output, "wb") as f:
            f.write(pixels.to_png_data())

    except Exception as e:
        print(e)
        exit(1)


if __name__ == "__main__":
    gds_file = ''
    if len(sys.argv) >= 2:
        gds_file = sys.argv[1]
    else:
        quit('Please provide a GDS file.')

    out_file = 'output.png'
    if len(sys.argv) >= 3:
        out_file = sys.argv[2]
    print(f"Input GDS file: {gds_file}")
    print(f"Output PNG file: {out_file}")

    render(gds_file, out_file)
    print("done.")

Now, run the utility function to generate an image of the layout of secret logic gate.

In [ ]:
!python gds_render.py secret.gds secret.png

Download and view the generated file ``secret.png``.


The power rails are labelled ``VPWR`` ($V_{DD}$) and ``VGND`` ($V_{SS}$ = ground). Start from the power rails, trace and list the layers until you reach any of the diffusion regions.

*Enter your answer here*

Estimate the area of the standard cell, in square micron ($\mu$m$^2$). *Hint: the narrowest feature on the layout is 150 nm.*

*Enter your answer here*


## Step 3: Identify logic function for the secret gate

What are the inputs and outputs of this gate? They are labelled on the metal 1 layer.

*Enter your answer here*

The gate is designed using the standard method with a pull-up network (PUN) and a pull-down network (PDN). PUN is built with only NMOS transistors and PDN is built with only PMOS transistors.

1. How many NMOS transistors and PMOS transistors are there in this standard cell?

*Enter your answer here*

2. Assume the transistors are either in series or in parallel with each other. Follow the way the transistors are connected, then deduce the function for the PUN and PDN respectively

PUN: *Enter your answer here*

PDN: *Enter your answer here*

3. Are PUN and PDN dual of each other? (duality: series ⇔ parallel, parallel ⇔ series)

*Enter your answer here*


## Step 4: Challenges of standard cell design

The gate is an example of a standard cell which may be used many times in a chip design. The following questions guide you to think about the challenges to design a standard cell.

Q4. Do you notice any wasted area in the cell? If yes, explain briefly how to reduce the waste. Why do we want to keep the cell area to be as small as possible?

*Enter your answer*

Q5. List two features that you find in the cell that makes it easy to use in combination with other cells in the library.

1st feature: *Enter your answer*

2nd feature: *Enter your answer*


If you have time, you may go back to the NAND gate studied in module 2. It is also an excellent example of standard cell.

---
**What did I learn?**

When you have studied and analysed a typical standard cell for chip design, add your thoughts in response to the following question. You will want to keep these, as you will find them useful when working on later modules and your final assignment. To get started, double click this cell to edit the text and answer the questions below.

**What would be the challenges to design and layout a logic gate as a standard cell? What are the changes for the cell under evolutions of the fabrication technologies?**

*Enter your answer here.*

---
You have now reached the end of the notebook for this week. Please return to Canvas to complete the rest of the work in module 3.

We will resume our work in this notebook next week, when we will be working through Module 4: Integrated circuit design.

# Module 4: Integrated circuit design

In this section you will work on your fourth module task. As you did before, please ensure that you are working in your own copy of the notebook and you have successfully run all the cells above.

In this task, we will take the HDL description of the 32-bit counter and use OpenLane EDA tools to produce a physical layout for it.

---

**What you need to do**

Firstly, set up Nix, OpenLane2 and SKY130 Open PDK in your Colab runtime environment, and then action the following steps:
1. Write the design file and install utility function.
1. Import the SKY130 PDK and start logic synthesis.
1. Action the logic synthesiser.
1. Complete floorplanning and PDN generation.
1. Place the design.
1. Route the clock tree and the design.
1. Sign-off and tape-out.


The following sections contain the code you need to complete these steps. Follow the boxes one by one.


## Step 1: Write the design file

Run the following code to create a verilog file for the 32-bit counter.

In [ ]:
%%writefile /content/counter32.v
/**
 * 32-Bit Binary Counter
 *
 * This module implements a basic up-counter that increments on the
 * positive edge of the clock (clk) when the enable signal (en) is high.
 * An asynchronous, active-high reset (reset) to initialise the count to 0.
 *
 * Inputs:
 * clk   : System clock signal.
 * reset : Asynchronous reset (active high).
 * en    : Enable signal for counting (active high).
 *
 * Output:
 * count : 32-bit register holding the current count value.
 */
module counter32 (
    input wire clk,
    input wire reset,
    input wire en,
    output reg [31:0] count
);

// The 'always' block is sensitive to the positive edge of the clock (posedge clk)
// and the positive edge of the reset (posedge reset) for asynchronous reset handling.
always @(posedge clk or posedge reset) begin
    if (reset) begin
        // Asynchronous reset: Immediately sets the counter to zero
        count <= 32'h0;
    end else if (en) begin
        // Synchronous increment: Increments the count on the positive edge of clk
        // if the enable signal is active (en = 1).
        count <= count + 1;
    end
    // If 'en' is low, the count holds its current value
end

endmodule

Run the next box to install a utility function for better pictures.

In [ ]:
%%writefile /content/openlane_ipynb/openlane/scripts/klayout/render.py
#!/usr/bin/env python3
# @markdown This makes ``display()`` to generate image of a higher resolution (default: 2500 pixels).
# More information about layers can be found at this
# [link](https://skywater-pdk.readthedocs.io/en/main/rules/layers.html). To hide the layer in the output image, comment off from the list.
#
# Copyright (c) 2021-2022 Efabless Corporation
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from typing import Tuple
import pya
import click
@click.command()
@click.option("-o", "--output", required=True)
@click.option("-l", "--input-lef", "input_lefs", multiple=True,)
@click.option( "-T", "--lyt", required=True, help="KLayout .lyt file",)
@click.option( "-P", "--lyp", required=True, help="KLayout .lyp file",)
@click.option( "-M", "--lym", required=True, help="KLayout .map (LEF/DEF layer map) file",)
@click.argument("input")
def render(input_lefs: Tuple[str, ...], output: str, lyt: str, lyp: str, lym: str, input: str,):
    image_res = 2500 # @param {key:"image resolution", type:"number"}
    # display ONLY the layers in the following list
    selected_layers = [
        "diff.",
        "dnwell.",
        "li1.",
        "licon1.",
        "mcon.",
        "met1.",
        "met2.",
        "met3.",
        "met4.",
        "met5.",
        #"vnpc.",
        #"vnsdm.",
        #"vnwell.",
        "poly.",
        "psdm.",
        "pwell.",
        "tap.",
        "via.",
        "via2.",
        "via3.",
        "via4."
    ]

    try:
        gds = input.endswith(".gds")

        # Load technology file
        tech = pya.Technology()
        tech.load(lyt)

        layout_options = None
        if not gds:
            layout_options = tech.load_layout_options
            layout_options.lefdef_config.map_file = lym
            layout_options.lefdef_config.macro_resolution_mode = 1
            layout_options.lefdef_config.read_lef_with_def = False
            layout_options.lefdef_config.lef_files = list(input_lefs)

        view = pya.LayoutView()
        view.load_layer_props(lyp)

        if gds:
            view.load_layout(input)
        else:
            view.load_layout(input, layout_options, lyt)

        view.max_hier()

        # control the visibility of the layers
        for layer in view.each_layer():
            layer.visible = False
            if '122/16@1' in layer.source: # grid and ruler
                layer.visible = True
            for t in selected_layers:
                if t in layer.name:
                    layer.visible = True

        pixels = view.get_pixels_with_options(image_res, image_res)

        with open(output, "wb") as f:
            f.write(pixels.to_png_data())

        # save an image of larger resolution
        pixels = view.get_pixels_with_options(image_res, image_res)
        with open("/content/openlane_run/output.png", "wb") as f:
            f.write(pixels.to_png_data())

    except Exception as e:
        print(e)
        exit(1)

if __name__ == "__main__":
    render()

## Step 2: Import SKY130 PDK

The OpenLane software will be controlled using a Python programme which will be listed below. The first snippet imports the OpenLane libraries and sets up the use of SKY130 PDK for our design ``counter32.v``.

In [ ]:
from openlane.steps import Step
from openlane.state import State
from openlane.config import Config

Config.interactive(
    "counter32",
    PDK="sky130A",
    CLOCK_PORT="clk",
    CLOCK_NET="clk",
    CLOCK_PERIOD=10,
    GPL_CELL_PADDING=2,
    DPL_CELL_PADDING=2,
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
)

There will be a long list of text mentioning the set of files loaded from the PDK.

## Step 3: Logic synthesis

Yosys is the logic synthesiser in OpenLane. The following code creates a Synthesis object and asks the logic synthesiser to work on the counter design.


In [ ]:
Synthesis = Step.factory.get("Yosys.Synthesis")
synthesis = Synthesis(
    VERILOG_FILES=["/content/counter32.v"],
    state_in=State(),
)
synthesis.start()

Look at the statistics near the end. Which cell is used most in the mapping?

*Enter your answer here*

Note the chip area at this stage.

Chip area = *Enter your answer here*

## Step 4: Floorplanning

The key tasks are to set the size of the die and create the vital power distribution network.

In [ ]:
Floorplan = Step.factory.get("OpenROAD.Floorplan")
floorplan = Floorplan(state_in=synthesis.state_out)
floorplan.start()

Look for die area in the log. What is the die area? How many rows of standard cell sites are allocated?

*Enter your answer here*

Run the code in the next box.

In [ ]:
TapEndcapInsertion = Step.factory.get("OpenROAD.TapEndcapInsertion")
tdi = TapEndcapInsertion(state_in=floorplan.state_out)
tdi.start()

IOPlacement = Step.factory.get("OpenROAD.IOPlacement")
ioplace = IOPlacement(state_in=tdi.state_out)
ioplace.start()
display(ioplace)

What difference did you notice?

*Enter your answer here*

In [ ]:
# generate power distribution network (PDN)
GeneratePDN = Step.factory.get("OpenROAD.GeneratePDN")
pdn = GeneratePDN(
    state_in=ioplace.state_out,
    FP_PDN_AUTO_ADJUST=True,
    FP_PDN_VPITCH=25,
    FP_PDN_HPITCH=25,
    FP_PDN_VOFFSET=5,
    FP_PDN_HOFFSET=5,
)
pdn.start()
display(pdn)

Which metal layers are used for the PDN? Suggest a reason.

*Enter your answer here*



## Step 5: Placement

The iterative global and detailed placements confirms the phyical positions of the standard cells.

In [ ]:
# Global placement
GlobalPlacement = Step.factory.get("OpenROAD.GlobalPlacement")
gpl = GlobalPlacement(state_in=pdn.state_out)
gpl.start()
display(gpl)

How many iterations (``Iter``) did the placer spend? What has changed along the iterations?

*Enter your answer here*

Do you notice the overlapping of the cells? Why would that be the case?

*Enter your answer here*

In [ ]:
# run detailed placement
DetailedPlacement = Step.factory.get("OpenROAD.DetailedPlacement")
dpl = DetailedPlacement(state_in=gpl.state_out)
dpl.start()
display(dpl)

Looking through the log, what is the percentage change in the half-perimeter wirelength (HPWL)?

*Enter your answer here*

Compare the total cell area with the one at the end of logic synthesis. How much more area is used?

*Enter your answer here*


## Step 6: Routing

The counter has a clock signal that runs to the flip-flops. Ahead of the global routing and detailed routing, the clock tree is synthesised. In the end, the whole design is fully routed.

In [ ]:
# clock tree synthesis
CTS = Step.factory.get("OpenROAD.CTS")
cts = CTS(state_in=dpl.state_out)
cts.start()

How many clock buffers has been added? What is the purpose of clock buffer?

*Enter your answer here*

In [ ]:
# Global routing
GlobalRouting = Step.factory.get("OpenROAD.GlobalRouting")
grt = GlobalRouting(state_in=cts.state_out)
grt.start()

What is the total wirelength? How many nets are routed?

*Enter your answer here*

In [ ]:
DetailedRouting = Step.factory.get("OpenROAD.DetailedRouting")
drt = DetailedRouting(state_in=grt.state_out)
drt.start()
display(drt)

Inspect the log and the routed design. How many optimisation iterations were used? What would be the main objective in these iterations?

*Enter your answer here*

## Step 7: Sign-off

The final touches to make sure the layout is correct and manufacturable.

In [ ]:
# Run fill insertion and display
FillInsertion = Step.factory.get("OpenROAD.FillInsertion")
fill = FillInsertion(state_in=drt.state_out)
fill.start()
display(fill)

Describe the main difference on the layout after fill insertion.

*Enter your answer here*

Next, we will extract parasitic RC from the layout and run the static timing analysis (STA).

In [ ]:
RCX = Step.factory.get("OpenROAD.RCX")
rcx = RCX(state_in=fill.state_out)
rcx.start()

STAPostPNR = Step.factory.get("OpenROAD.STAPostPNR")
sta_post_pnr = STAPostPNR(state_in=rcx.state_out)
sta_post_pnr.start()

In [ ]:
# Generate the final GDSII layout
StreamOut = Step.factory.get("KLayout.StreamOut")
gds = StreamOut(state_in=sta_post_pnr.state_out)
gds.start()
display(gds)

Please save a copy of the picture of this final layout for your final assignment.

In [ ]:
# Final design rule check
DRC = Step.factory.get("Magic.DRC")
drc = DRC(state_in=gds.state_out)
drc.start()

Is there any DRC? (Yes/No)

*Enter your answer here*

In [ ]:
# extract the SPICE netlist of devices and connections from the layout
SpiceExtraction = Step.factory.get("Magic.SpiceExtraction")
spx = SpiceExtraction(state_in=drc.state_out)
spx.start()
# Run layout-vs-schematic (LVS)
LVS = Step.factory.get("Netgen.LVS")
lvs = LVS(state_in=spx.state_out)
lvs.start()

Does the layout match with the schematic? (Yes/No)

*Enter your answer here*

If your answer is **YES**, then the layout is ready for a tape-out to the foundry! That's the end of the EDA journey.

---
**What did I learn?**

When you have completed the generation of a physical layout of a 32-bit counter, add your thoughts in response to the following question. You will want to keep these, as you will find them useful when working on later modules and your final assignment. To get started, double click this cell and edit the text.

**What are your biggest challenges in working with EDA tools? Do you find them user friendly?**

*Enter your answer here.*

---
You have now reached the end of the notebook for this week. Please return to Canvas to complete the rest of the work in module 4.

We will resume our work in this notebook next week, when we will be working through Module 5: Assembly, testing and packaging.

# Module 5: Assembly, testing and packaging
In this section you will work on your fifth module task. Before attempting it, please ensure that you are working in your own copy of the notebook and you have successfully run all the cells above.

In this task, we are going to prepare an inverter design for assembly and packaging using the popular wire bonding technology.

---

**What you need to do**

Firstly, set up Nix, OpenLane2 and SKY130 Open PDK in your Colab runtime environment, and then action the following steps:
1. Write the design file and install the utility function.
1. Run automatic floorplanning.
1. Run manual floorplanning with fixed die size.
1. Work out the expected die size with I/O pads inserted.

The following sections contain the code and packages you need to complete these steps.

## Step 1: Write the design file

Run the following code to create a verilog file for a 32-bit inverter.

In [ ]:
%%writefile /content/inverter32.v
module inverter32 (
    input  wire [31:0] a_in,
    output wire [31:0] y_out
);

    assign y_out = ~a_in;
endmodule

Run the next box to install a utility function to visualise the layout.

In [ ]:
%%writefile /content/openlane_ipynb/openlane/scripts/klayout/render.py
#!/usr/bin/env python3
# @markdown This makes ``display()`` to generate image of a higher resolution. Only the metal layers are shown (others are hidden in this task).
# More information about layers can be found at this
# [link](https://skywater-pdk.readthedocs.io/en/main/rules/layers.html). To hide the layer in the output image, comment off from the list.
#
# Copyright (c) 2021-2022 Efabless Corporation
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from typing import Tuple
import pya
import click
@click.command()
@click.option("-o", "--output", required=True)
@click.option("-l", "--input-lef", "input_lefs", multiple=True,)
@click.option( "-T", "--lyt", required=True, help="KLayout .lyt file",)
@click.option( "-P", "--lyp", required=True, help="KLayout .lyp file",)
@click.option( "-M", "--lym", required=True, help="KLayout .map (LEF/DEF layer map) file",)
@click.argument("input")
def render(input_lefs: Tuple[str, ...], output: str, lyt: str, lyp: str, lym: str, input: str,):
    image_res = 1000 # @param {key:"image resolution", type:"number"}
    # display ONLY the layers in the following list
    selected_layers = [
        #"diff.",
        #"dnwell.",
        #"li1.",
        #"licon1.",
        #"mcon.",
        "met1.",
        "met2.",
        "met3.",
        "met4.",
        "met5.",
        #"vnpc.",
        #"vnsdm.",
        #"vnwell.",
        #"poly.",
        #"psdm.",
        #"pwell.",
        #"tap.",
        "via.",
        "via2.",
        "via3.",
        "via4."
    ]

    try:
        gds = input.endswith(".gds")

        # Load technology file
        tech = pya.Technology()
        tech.load(lyt)

        layout_options = None
        if not gds:
            layout_options = tech.load_layout_options
            layout_options.lefdef_config.map_file = lym
            layout_options.lefdef_config.macro_resolution_mode = 1
            layout_options.lefdef_config.read_lef_with_def = False
            layout_options.lefdef_config.lef_files = list(input_lefs)

        view = pya.LayoutView()
        view.load_layer_props(lyp)

        if gds:
            view.load_layout(input)
        else:
            view.load_layout(input, layout_options, lyt)

        view.max_hier()

        # control the visibility of the layers
        for layer in view.each_layer():
            layer.visible = False
            if '122/16@1' in layer.source: # grid and ruler
                layer.visible = True
            for t in selected_layers:
                if t in layer.name:
                    layer.visible = True

        pixels = view.get_pixels_with_options(image_res, image_res)

        with open(output, "wb") as f:
            f.write(pixels.to_png_data())

        # save an image of larger resolution
        pixels = view.get_pixels_with_options(image_res, image_res)
        with open("/content/openlane_run/output.png", "wb") as f:
            f.write(pixels.to_png_data())

    except Exception as e:
        print(e)
        exit(1)

if __name__ == "__main__":
    render()

## Step 2: First trial – automatic floorplan

We will set up OpenLane as usual, taking all default settings.

In [ ]:
from openlane.steps import Step
from openlane.state import State
from openlane.config import Config

Config.interactive(
    "inverter32",
    PDK="sky130A",
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
)

Next, we will synthesise the circuit.

In [ ]:
Synthesis = Step.factory.get("Yosys.Synthesis")
synthesis = Synthesis(
    VERILOG_FILES=["/content/inverter32.v"],
    state_in=State(),
)
synthesis.start()

As we expected, there are 32 inverter standard cells. How many public wires and public wire bits are there? Can you guess their meaning?

*Enter your answer here*

Next, we will proceed with floorplanning. Remember that the tool is set to find the smallest possible size of the die.

In [ ]:
Floorplan = Step.factory.get("OpenROAD.Floorplan")
floorplan = Floorplan(state_in=synthesis.state_out)
floorplan.start()

Look for die area in the log. What is the dimension of the die?

*Enter your answer here*

Next, we will insert tap cells and end cap cells. Then randomly assign I/O pins.

In [ ]:
TapEndcapInsertion = Step.factory.get("OpenROAD.TapEndcapInsertion")
tdi = TapEndcapInsertion(state_in=floorplan.state_out)
tdi.start()

IOPlacement = Step.factory.get("OpenROAD.IOPlacement")
ioplace = IOPlacement(state_in=tdi.state_out)
ioplace.start()
display(ioplace)

You should receive an error at this box. What is the explanation of the **ERROR** from the floorplan?

*Enter your answer here*

## Step 3: Second trial – floorplan with fixed die size

Based on the suggestion from the tools, provide a new dimension for the die to fit all I/O pins. Enter your answers (new width, new height) in the form below.



In [ ]:
from openlane.steps import Step
from openlane.state import State
from openlane.config import Config

new_die_width = 10 # @param {key:"New die width", type:"number"}
new_die_height = 10 # @param {key:"New die height", type:"number"}
Config.interactive(
    "inverter32",
    PDK="sky130A",
    DIE_AREA=[0, 0, new_die_width, new_die_height],
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
)

Let's synthesise again.

In [ ]:
Synthesis = Step.factory.get("Yosys.Synthesis")
synthesis = Synthesis(
    VERILOG_FILES=["/content/inverter32.v"],
    state_in=State(),
)
synthesis.start()

This time, we ask the floorplanner to respect the die size supplied with the option ``FP_SIZING`` as **absolute**. Run the floorplanning again.

In [ ]:
Floorplan = Step.factory.get("OpenROAD.Floorplan")
floorplan = Floorplan(FP_SIZING="absolute", state_in=synthesis.state_out)
floorplan.start()

Inspect the output on die area and confirm that the new dimension is in place.

What is the core area in μm${}^2$?

*Enter your answer here*

Next, we will try again with the I/O placement.

In [ ]:
TapEndcapInsertion = Step.factory.get("OpenROAD.TapEndcapInsertion")
tdi = TapEndcapInsertion(state_in=floorplan.state_out)
tdi.start()

IOPlacement = Step.factory.get("OpenROAD.IOPlacement")
ioplace = IOPlacement(state_in=tdi.state_out)
ioplace.start()
display(ioplace)

If it is **not successful**, return to the first box in **Step 3** and enter a bigger width and height.

Which metal layer(s) is/are used for the I/O pins?

*Enter your answer here*

Estimate the pitch of the I/O pins. The pitch is the distance between two pins.

*Enter your answer here*

We will proceed with the remaining steps of the physical implementation.

In [ ]:
# generate power distribution network (PDN)
GeneratePDN = Step.factory.get("OpenROAD.GeneratePDN")
pdn = GeneratePDN(
    state_in=ioplace.state_out,
    FP_PDN_AUTO_ADJUST=True,
    FP_PDN_VPITCH=25,
    FP_PDN_HPITCH=25,
    FP_PDN_VOFFSET=5,
    FP_PDN_HOFFSET=5,
)
pdn.start()

# Global placement
GlobalPlacement = Step.factory.get("OpenROAD.GlobalPlacement")
gpl = GlobalPlacement(state_in=pdn.state_out)
gpl.start()

# run detailed placement
DetailedPlacement = Step.factory.get("OpenROAD.DetailedPlacement")
dpl = DetailedPlacement(state_in=gpl.state_out)
dpl.start()

display(dpl)

Next, we will route the design.

In [ ]:
# Global routing
GlobalRouting = Step.factory.get("OpenROAD.GlobalRouting")
grt = GlobalRouting(state_in=dpl.state_out)
grt.start()

DetailedRouting = Step.factory.get("OpenROAD.DetailedRouting")
drt = DetailedRouting(state_in=grt.state_out)
drt.start()
display(drt)

Inspect the log and the routed design. What is the final total wirelength?

*Enter your answer here*

You may consider this as the length required to connect the logic circuit with the I/O pins.

Please save a copy of the picture of this final layout for your final assignment.

## Step 4: Challenges of inserting I/O pads

From the task above, we have successfully assigned I/O pins around the die, but without any I/O pads. I/O pads are necessary to provide electrical protection and an area for wire bonding.

A typical in-line pad pitch required for 0.18 μm technology is [70 μm](https://www.onsemi.com/PowerSolutions/content.do?id=16679). What would be the dimension of the die if in-line pads are added?

*Enter your answer here*

How many percentage of the area is used as the core in this case?

*Enter your answer here*


---

**What did I learn?**

When you have completed the manual floorplanning to fit in I/O pins and pads, add your thoughts in response to the following question. You will want to keep these, as you will find them useful when working on later modules and your final assignment. To get started, double click this cell and edit the text.

**How does assembly and packaging affect chip design? What are the necessary steps during design to prepare the chip for final packaging?**

*Enter your answer here.*


---
You have now reached the end of the notebook for this week. Please return to Canvas to complete the rest of the work in module 5.
